In [ ]:
import torch
import math

In [ ]:
# -------------------------------------------------
# 1️⃣ Input embeddings (6 tokens, 3-dim each)
# -------------------------------------------------
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your
        [0.55, 0.87, 0.66],  # journey
        [0.57, 0.85, 0.64],  # starts
        [0.22, 0.58, 0.33],  # with
        [0.77, 0.25, 0.10],  # one
        [0.05, 0.80, 0.55],  # step
    ],
    dtype=torch.float32
)

In [ ]:
# -------------------------------------------------
# 2️⃣ Define dimensions
# -------------------------------------------------
d_in = inputs.shape[1]   # 3
d_out = 2                # projection dimension

torch.manual_seed(123)

In [ ]:
# -------------------------------------------------
# 3️⃣ Initialize projection matrices
# -------------------------------------------------
W_query = torch.rand(d_in, d_out)
W_key   = torch.rand(d_in, d_out)
W_value = torch.rand(d_in, d_out)

In [ ]:
# -------------------------------------------------
# 4️⃣ Compute Q, K, V
# -------------------------------------------------
queries = inputs @ W_query     # (6,2)
keys    = inputs @ W_key       # (6,2)
values  = inputs @ W_value     # (6,2)

In [ ]:
# -------------------------------------------------
# 5️⃣ Compute Attention Scores
# -------------------------------------------------
attn_scores = queries @ keys.T   # (6,6)

In [ ]:
# -------------------------------------------------
# 6️⃣ Scale
# -------------------------------------------------
scale = math.sqrt(d_out)
attn_scores = attn_scores / scale

In [ ]:
# -------------------------------------------------
# 7️⃣ Softmax → Attention Weights
# -------------------------------------------------
attn_weights = torch.softmax(attn_scores, dim=-1)

In [ ]:
# -------------------------------------------------
# 8️⃣ Compute Context Vectors
# -------------------------------------------------
context_vectors = attn_weights @ values   # (6,2)

In [ ]:
# -------------------------------------------------
# 9️⃣ Print Results
# -------------------------------------------------
print("Queries shape:", queries.shape)
print("Keys shape:", keys.shape)
print("Attention Scores shape:", attn_scores.shape)
print("Attention Weights shape:", attn_weights.shape)
print("Context Vectors shape:", context_vectors.shape)

print("\nContext Vectors:\n", context_vectors)

Queries shape: torch.Size([6, 2])
Keys shape: torch.Size([6, 2])
Attention Scores shape: torch.Size([6, 6])
Attention Weights shape: torch.Size([6, 6])
Context Vectors shape: torch.Size([6, 2])

Context Vectors:
 tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])


In [ ]:
import torch
import torch.nn as nn
import math

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        # Learnable projection matrices
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

        self.d_out = d_out

    def forward(self, x):
        """
        x shape: (seq_len, d_in)
        """

        # Project inputs to Q, K, V
        queries = x @ self.W_query     # (seq_len, d_out)
        keys    = x @ self.W_key       # (seq_len, d_out)
        values  = x @ self.W_value     # (seq_len, d_out)

        # Compute attention scores
        attn_scores = queries @ keys.T   # (seq_len, seq_len)

        # Scale
        attn_scores = attn_scores / math.sqrt(self.d_out)

        # Softmax
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # Compute context vectors
        context = attn_weights @ values  # (seq_len, d_out)

        return context, attn_weights

In [ ]:
torch.manual_seed(123)
sa_V1 = SelfAttention_v1(d_in,d_out)
print(sa_V1(inputs))

(tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>), tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]],
       grad_fn=<SoftmaxBackward0>))


In [ ]:
import torch
import torch.nn as nn
import math

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        # Linear layers for Q, K, V
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.d_out = d_out

    def forward(self, x):
        """
        x shape: (seq_len, d_in)
        or (batch_size, seq_len, d_in)
        """

        # Compute Q, K, V
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        # Attention scores
        attn_scores = queries @ keys.transpose(-2, -1)

        # Scale
        attn_scores = attn_scores / math.sqrt(self.d_out)

        # Softmax
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # Context vectors
        context = attn_weights @ values

        return context, attn_weights

In [ ]:
torch.manual_seed(123)
sa_V2 = SelfAttention_v1(d_in,d_out)
print(sa_V2(inputs))

(tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>), tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>))


### Causal Attention

In [ ]:
queries = sa_V2.W_query(inputs)
keys = sa_V2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1]**0.5,
    dim=1
)

print(attn_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
context_length = attn_scores.shape[0]

mask_simple = torch.tril(torch.ones(context_length, context_length))

print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


### Data leakage problem because when cal context emb already the embedding in the un masked places are influenced by other vectors

In [ ]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
masked_simple_norm = masked_simple/row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


### DO by Attn score ---> Apply masked infinity ---> Softmax

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length),diagonal=1)
masked = attn_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


In [ ]:
attn_weights = torch.softmax(
    masked / keys.shape[-1]**0.5,
    dim=1
)

In [ ]:
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


### Adding Dropout

In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6380, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.5085, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.0000, 0.3869, 0.0000, 0.0000],
        [0.0000, 0.3418, 0.3413, 0.3308, 0.3249, 0.0000]],
       grad_fn=<MulBackward0>)


In [ ]:
batch = torch.stack((inputs,inputs),dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


### Implementing in Class

In [ ]:
import torch
import torch.nn as nn
import math

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.1, qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.d_out = d_out

        # Causal mask (lower triangular)
        mask = torch.tril(torch.ones(context_length, context_length))
        self.register_buffer("mask", mask)

    def forward(self, x):
        """
        x: (batch_size, seq_len, d_in)
        """

        B, T, C = x.shape

        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        # (B, T, T)
        attn_scores = queries @ keys.transpose(-2, -1)
        attn_scores = attn_scores / math.sqrt(self.d_out)

        # Apply mask
        attn_scores = attn_scores.masked_fill(
            self.mask[:T, :T] == 0, float('-inf')
        )

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = attn_weights @ values

        return context, attn_weights

In [ ]:
print(d_in)

3


In [ ]:
print(d_out)

2


In [ ]:
context_vecs, attn_weights = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


In [ ]:
print(context_vecs)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


### Multi Head Attention

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        # Only take context output from each head
        head_outputs = [head(x)[0] for head in self.heads]

        # Concatenate on last dimension
        return torch.cat(head_outputs, dim=-1)

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in,d_out = 3,2
mha = MultiHeadAttentionWrapper(d_in,d_out,context_length,0.0,num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print(context_vecs.shape)

### Implementing Multi head with weight splits

In [ ]:
import torch
import torch.nn as nn

class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        self.num_heads = num_heads
        self.d_out = d_out

        # Create multiple independent attention heads
        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )

        # Final projection after concatenation
        self.out_proj = nn.Linear(num_heads * d_out, num_heads * d_out)

    def forward(self, x):
        """
        x: (B, T, d_in)
        """

        head_outputs = []
        all_attn_weights = []

        for head in self.heads:
            context, attn_weights = head(x)
            head_outputs.append(context)
            all_attn_weights.append(attn_weights)

        # Concatenate along feature dimension
        # Each context: (B, T, d_out)
        # After concat: (B, T, num_heads * d_out)
        multi_head_output = torch.cat(head_outputs, dim=-1)

        # Final linear projection
        output = self.out_proj(multi_head_output)

        return output, all_attn_weights